# Experiment 07 — MIN Frequency Response

Load the recorded Experiment 07 CSV and visualize the reference, finite-window, causal endpoint, and tail estimates.

In [ ]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "experiments" / "results"
rows = list(csv.DictReader((RESULTS / "07_frequency_response_results.csv").open()))
summary = json.loads((RESULTS / "07_frequency_response_summary.json").read_text())
kernels = list(dict.fromkeys(r["kernel"] for r in rows))

def col(subset, key):
    return np.array([float(r[key]) for r in subset])

In [ ]:
for kernel in kernels:
    subset = sorted([r for r in rows if r["kernel"] == kernel], key=lambda r: float(r["omega_rad_s"]))
    w = col(subset, "omega_rad_s")
    ref = col(subset, "H_ref_magnitude")
    win = col(subset, "H_window_magnitude")
    endpoint = col(subset, "H_endpoint_magnitude")
    tail = col(subset, "H_tail_magnitude")
    plt.figure()
    plt.loglog(w, ref, label="reference")
    plt.loglog(w, win, marker="o", label="finite window")
    plt.loglog(w, endpoint, linestyle="--", label="endpoint")
    plt.loglog(w, tail, linestyle=":", label="tail")
    plt.xlabel("Angular frequency ω [rad/s]")
    plt.ylabel("|H(ω)|")
    plt.title(kernel + " — magnitude response")
    plt.legend()
    plt.grid(True, which="both", alpha=0.25)
    plt.show()

In [ ]:
for kernel in kernels:
    subset = sorted([r for r in rows if r["kernel"] == kernel], key=lambda r: float(r["omega_rad_s"]))
    w = col(subset, "omega_rad_s")
    ref_phase = np.unwrap(col(subset, "H_ref_phase_rad"))
    plt.figure()
    plt.semilogx(w, np.degrees(ref_phase), marker="o", label="reference")
    plt.xlabel("Angular frequency ω [rad/s]")
    plt.ylabel("Reference phase [deg]")
    plt.title(kernel + " — phase response")
    plt.legend()
    plt.grid(True, which="both", alpha=0.25)
    plt.show()

In [ ]:
plt.figure()
for kernel in kernels:
    subset = sorted([r for r in rows if r["kernel"] == kernel], key=lambda r: float(r["omega_rad_s"]))
    w = col(subset, "omega_rad_s")
    err = col(subset, "window_vs_reference_rel_error")
    plt.loglog(w, np.maximum(err, 1e-16), marker="o", label=kernel)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Finite-window / reference relative error")
plt.title("Observation-horizon effect")
plt.legend()
plt.grid(True, which="both", alpha=0.25)
plt.show()

## Interpretation

The endpoint estimator is an implementation check and follows the finite-window transform to numerical precision. The power-law case shows substantial observation-horizon dependence, especially at low frequency, which is the main phenomenon to carry into the communication-signal phase.